# Forecasting, Riesgo y Asset Allocation — Modelo de Seis Factores de Fama-French

**Tesis de Magíster — Universidad Diego Portales**  
**Autores:** Álvaro Gatica, Fernanda Rojas, Luis Pizarro, Cristian Illanes  
**Notebook 01:** Descarga y Exploración de Datos

---

### Objetivo de este notebook

Este notebook es el punto de partida del proyecto. Su función es:

1. Descargar los **factores Fama-French 6 (FF6)** desde la base de datos pública de Kenneth French.
2. Descargar los **precios históricos ajustados** de un universo de activos financieros.
3. Calcular **retornos** diarios y mensuales.
4. Explorar **estadísticas descriptivas**, correlaciones y distribuciones de retornos.
5. Exportar los datos limpios para ser usados en los notebooks siguientes.

> **Nota:** Todo el código, comentarios y documentación de este proyecto están escritos en **español**.

---
## 0. Instalación de dependencias

Ejecutar **una sola vez** al configurar el entorno por primera vez.  
Luego comentar esta celda para no reinstalar en cada ejecución.

In [ ]:
# pip install: instala paquetes de Python desde el repositorio PyPI.
# Cada librería tiene un rol específico en el proyecto (ver sección de imports).
# Descomenta la línea siguiente si es la primera vez que ejecutas este proyecto:

# !pip install pandas numpy pandas_datareader yfinance statsmodels arch cvxpy scipy matplotlib seaborn jupyter

---
## 1. Importación de librerías y configuración global

### ¿Qué hace cada librería?

| Librería | Rol en el proyecto |
|---|---|
| `pandas` | Manipulación de datos tabulares (DataFrames y Series). Es la columna vertebral del proyecto: almacena precios, retornos, factores y resultados. |
| `numpy` | Operaciones matemáticas y álgebra lineal sobre arreglos numéricos. Se usa para cálculos vectorizados (medias, varianzas, matrices de covarianza). |
| `matplotlib` | Librería base para gráficos en Python. Permite crear figuras, ejes, líneas, histogramas y personalizar cada elemento visual. |
| `matplotlib.dates` | Módulo de matplotlib especializado en formatear fechas en los ejes de los gráficos de series de tiempo. |
| `seaborn` | Librería de visualización construida sobre matplotlib. Simplifica gráficos estadísticos como mapas de calor y distribuciones. |
| `warnings` | Módulo estándar de Python para controlar mensajes de advertencia del intérprete y librerías externas. |
| `pandas_datareader` | Permite descargar datos financieros desde fuentes externas (FRED, Fama-French, Nasdaq, etc.) directamente a DataFrames de pandas. |
| `yfinance` | Cliente no oficial de Yahoo Finance para descargar precios históricos, dividendos y splits de miles de activos financieros. |
| `datetime` | Módulo estándar de Python para trabajar con fechas y horas. Se usa para definir el rango temporal del análisis. |
| `scipy.stats` | Módulo de estadística científica: distribuciones de probabilidad, pruebas de hipótesis (Jarque-Bera, Shapiro-Wilk) y funciones de densidad. |

In [ ]:
# ── Librerías de manipulación de datos ────────────────────────────────────
import pandas as pd          # Estructuras de datos tabulares (DataFrame, Series)
import numpy as np           # Álgebra lineal y operaciones numéricas vectorizadas

# ── Librerías de visualización ────────────────────────────────────────────
import matplotlib.pyplot as plt        # Motor principal de gráficos
import matplotlib.dates as mdates      # Formateador de fechas en ejes de gráficos
import seaborn as sns                  # Gráficos estadísticos de alto nivel

# ── Librería para suprimir advertencias ───────────────────────────────────
import warnings                        # Control de mensajes de advertencia del sistema

# ── Librerías de descarga de datos financieros ───────────────────────────
import pandas_datareader.data as web   # Descarga datos desde Fama-French, FRED, etc.
import yfinance as yf                  # Descarga precios históricos desde Yahoo Finance

# ── Librerías estándar de Python ──────────────────────────────────────────
from datetime import datetime          # Manejo de fechas para definir el rango temporal
from scipy import stats as sp_stats    # Pruebas estadísticas y funciones de densidad

# ── Configuración global del notebook ────────────────────────────────────
warnings.filterwarnings('ignore')      # Silencia advertencias de librerías (ej. deprecaciones)

# Formato de números flotantes: muestra 4 decimales en las tablas de pandas
pd.set_option('display.float_format', '{:.4f}'.format)

# Tamaño predeterminado de todos los gráficos (ancho x alto en pulgadas)
plt.rcParams['figure.figsize'] = (12, 5)

# Activa la cuadrícula de fondo en todos los gráficos
plt.rcParams['axes.grid'] = True

# Transparencia de la cuadrícula (0 = invisible, 1 = sólido)
plt.rcParams['grid.alpha'] = 0.3

# ── Parámetros globales del análisis ─────────────────────────────────────
START = '2000-01-01'                            # Fecha de inicio del período de análisis
END   = datetime.today().strftime('%Y-%m-%d')   # Fecha de fin: hoy (formato AAAA-MM-DD)

print(f'Período de análisis: {START} → {END}')

---
## 2. Descarga de Factores Fama-French 6 (FF6)

### ¿Qué son los factores Fama-French?

Los factores FF6 son variables que explican los retornos de los activos financieros más allá del riesgo de mercado (CAPM). Cada factor captura una prima de riesgo sistemática observada empíricamente:

| Factor | Nombre completo | Interpretación económica |
|---|---|---|
| **Mkt-RF** | Market minus Risk-Free | Prima de riesgo del mercado accionario sobre la tasa libre de riesgo |
| **SMB** | Small Minus Big | Acciones pequeñas tienden a superar a las grandes en el largo plazo |
| **HML** | High Minus Low | Acciones de valor (alto P/B) tienden a superar a las de crecimiento |
| **RMW** | Robust Minus Weak | Empresas con alta rentabilidad operacional superan a las menos rentables |
| **CMA** | Conservative Minus Aggressive | Empresas que invierten conservadoramente superan a las que invierten agresivamente |
| **WML** | Winners Minus Losers | Activos con buen desempeño reciente tienden a seguir rindiendo bien (momentum) |
| **RF** | Risk-Free Rate | Tasa libre de riesgo mensual (rendimiento de T-Bills estadounidenses) |

**Fuente:** [Kenneth French Data Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html)  
**Método de descarga:** `pandas_datareader` accede a la librería de French directamente.

In [ ]:
# ── Descarga de los 5 factores base + RF ──────────────────────────────────
# 'F-F_Research_Data_5_Factors_2x3': nombre del dataset en la base de datos de French
# 'famafrench': nombre de la fuente dentro de pandas_datareader
# [0]: el dataset devuelve una lista; [0] es la tabla mensual (la más común)
# / 100: los datos vienen en porcentaje (ej. 0.75 = 0.75%); dividir convierte a decimal (0.0075)
ff5_monthly = web.DataReader(
    'F-F_Research_Data_5_Factors_2x3',   # Nombre del archivo en la librería de French
    'famafrench',                          # Fuente: Kenneth French Data Library
    start=START                            # Fecha de inicio de la descarga
)[0] / 100                                 # [0] = tabla mensual; /100 = pasar de % a decimal

# ── Descarga del factor Momentum (6to factor) ─────────────────────────────
# El momentum NO está en el dataset de los 5 factores; se descarga por separado
mom_monthly = web.DataReader(
    'F-F_Momentum_Factor',    # Archivo separado del momentum en la librería de French
    'famafrench',
    start=START
)[0] / 100

# Renombrar la columna: viene como 'Mom' pero usaremos 'WML' (Winners Minus Losers)
mom_monthly.columns = ['WML']

# ── Unir los 5 factores + momentum en un único DataFrame FF6 ──────────────
# .join(): une dos DataFrames por el índice (en este caso, por fecha)
# how='inner': conserva solo las fechas presentes en AMBOS datasets (intersección)
ff6 = ff5_monthly.join(mom_monthly, how='inner')

# Convertir el índice de período mensual (Period) a fecha (Timestamp)
# to_timestamp(): convierte cada período al primer día del mes correspondiente
# pd.to_datetime(): asegura que el tipo sea datetime64 de pandas
ff6.index = pd.to_datetime(ff6.index.to_timestamp())

# ── Resumen ───────────────────────────────────────────────────────────────
print(f'Shape: {ff6.shape}')   # (filas = meses, columnas = 7 factores + RF)
print(f'Período: {ff6.index[0].date()} → {ff6.index[-1].date()}')
ff6.tail()    # Mostrar las últimas 5 filas para verificar la descarga

In [ ]:
# ── Estadísticas descriptivas de los factores ─────────────────────────────
# Los factores se expresan en frecuencia mensual; los analizamos en escala anual
# para facilitar la interpretación económica

# Separar solo los 6 factores (excluir RF que es la tasa libre de riesgo)
factores = [c for c in ff6.columns if c != 'RF']   # Lista con los nombres de los 6 factores
ff6_f = ff6[factores]                              # Sub-DataFrame solo con los factores

stats_ff6 = pd.DataFrame({
    # Media mensual × 12 = retorno promedio anualizado
    'Media anual':       ff6_f.mean() * 12,

    # Desviación estándar mensual × √12 = volatilidad anualizada
    # (La raíz de 12 viene de la propiedad de la varianza: Var_anual = Var_mensual × 12)
    'Volatilidad anual': ff6_f.std() * np.sqrt(12),

    # Sharpe = retorno anual / volatilidad anual (mide eficiencia del factor)
    'Sharpe':            (ff6_f.mean() * 12) / (ff6_f.std() * np.sqrt(12)),

    # Sesgo (skewness): 0 = simétrico, negativo = cola izquierda más pesada
    'Skewness':          ff6_f.skew(),

    # Curtosis (kurtosis): 0 = normal, positivo = colas más gruesas que la normal (leptocúrtica)
    'Kurtosis':          ff6_f.kurt()
})
stats_ff6

In [ ]:
# ── Retornos acumulados de cada factor ────────────────────────────────────
# Retorno acumulado = producto de (1 + retorno mensual) en el tiempo
# Interpreta: si el factor Mkt-RF tiene valor 3.0 en 2020, significa que
# $1 invertido en ese factor en el año 2000 valió $3 en 2020

# (1 + ff6_f): transforma retornos (ej. -0.03) en factores de crecimiento (ej. 0.97)
# .cumprod(): multiplica acumulativamente todos los valores hasta cada fecha
cumret = (1 + ff6_f).cumprod()

fig, ax = plt.subplots(figsize=(14, 6))    # Crear figura y eje con tamaño personalizado

# Graficar cada factor como una línea independiente
for col in cumret.columns:
    ax.plot(cumret.index, cumret[col], label=col, linewidth=1.5)

ax.set_title('Retornos Acumulados — Factores Fama-French 6', fontsize=14)
ax.set_ylabel('Valor acumulado (base = 1.0 en el inicio)')   # Eje Y: dólares por $1 invertido

# Marcar una fecha mayor en el eje X cada 2 años para no sobrecargar el gráfico
ax.xaxis.set_major_locator(mdates.YearLocator(2))

ax.legend(loc='upper left')    # Leyenda con los nombres de los factores
plt.tight_layout()             # Ajusta márgenes automáticamente para evitar recortes
plt.show()

---
## 3. Universo de Activos — Descarga de Precios

### Criterio de selección

Se utiliza un conjunto representativo de acciones del **S&P 500** clasificadas por sector, más el ETF **SPY** como benchmark.  
Esta selección puede modificarse en la variable `TICKERS` según el universo final definido en la tesis.

### Precio ajustado vs. precio de cierre

Se descarga el **precio ajustado** (`auto_adjust=True`), que incorpora retroactivamente el efecto de dividendos y splits. Esto permite calcular retornos totales reales sin sesgos por eventos corporativos.

In [ ]:
# ── Definición del universo de activos ────────────────────────────────────
# TICKERS: lista de símbolos bursátiles (tickers) de Yahoo Finance
# Modificar esta lista para cambiar el universo de activos del análisis
TICKERS = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META',   # Tecnología (sector de mayor peso en S&P500)
    'JPM',  'BAC',  'GS',                        # Servicios financieros (bancos)
    'JNJ',  'PFE',  'UNH',                       # Salud (farmacéuticas y seguros)
    'XOM',  'CVX',                               # Energía (petróleo y gas)
    'WMT',  'PG',   'KO',                        # Consumo básico (defensivos, baja volatilidad)
    'SPY'                                         # ETF benchmark del S&P 500 completo
]

# ── Descarga de precios ajustados desde Yahoo Finance ─────────────────────
# yf.download(): descarga una tabla de precios OHLCV para múltiples activos
# TICKERS: lista de símbolos a descargar
# start, end: rango de fechas del análisis
# auto_adjust=True: descarga precio ajustado (incorpora dividendos y splits)
# progress=False: desactiva la barra de progreso en la consola
# ['Close']: selecciona solo la columna de precio de cierre (ajustado)
raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True, progress=False)['Close']

# Asegurar que el índice sea de tipo datetime64 (por compatibilidad con pandas)
raw.index = pd.to_datetime(raw.index)

print(f'Shape: {raw.shape}')   # (filas = días hábiles, columnas = nº de activos)
print(f'Período: {raw.index[0].date()} → {raw.index[-1].date()}')
raw.tail()    # Mostrar los últimos precios descargados

In [ ]:
# ── Cálculo de retornos ───────────────────────────────────────────────────

# Retornos diarios simples: r_t = (P_t - P_{t-1}) / P_{t-1}
# .pct_change(): calcula la variación porcentual respecto a la fila anterior
# .dropna(): elimina la primera fila (que queda como NaN al no tener precio anterior)
ret_daily = raw.pct_change().dropna()

# Retornos mensuales: se toma el precio de cierre del último día de cada mes
# .resample('ME'): agrupa los datos por mes (ME = Month End = último día del mes)
# .last(): toma el último valor disponible de cada grupo (precio de cierre del mes)
# .pct_change(): calcula el retorno mensual respecto al mes anterior
# .dropna(): elimina el primer mes (sin precio anterior)
ret_monthly = raw.resample('ME').last().pct_change().dropna()

print(f'Retornos diarios   — Shape: {ret_daily.shape}')    # (~5000 días × 17 activos)
print(f'Retornos mensuales — Shape: {ret_monthly.shape}')  # (~280 meses × 17 activos)

---
## 4. Estadísticas Descriptivas de los Activos

Se calculan métricas de riesgo y retorno para cada activo usando frecuencia mensual, alineada con los factores FF6.  
Todas las métricas se **anualizan** para facilitar la comparación con benchmarks y con la literatura.

In [ ]:
# ── Alineación temporal entre retornos y factores ─────────────────────────
# Los retornos mensuales y los factores FF6 deben tener exactamente las mismas fechas
# .intersection(): devuelve solo las fechas que están presentes en AMBOS índices
common_idx = ret_monthly.index.intersection(ff6.index)

# Filtrar cada DataFrame para que solo contenga las fechas comunes
ret_m = ret_monthly.loc[common_idx]   # Retornos mensuales alineados
ff6_m = ff6.loc[common_idx]           # Factores FF6 alineados
Rf_m  = ff6_m['RF']                   # Tasa libre de riesgo mensual (T-Bills EE.UU.)

n = 12    # Factor de anualización: 12 meses en un año

# ── Tabla de estadísticas anualizadas ─────────────────────────────────────
stats_activos = pd.DataFrame({

    # Retorno promedio anualizado: E[r_mensual] × 12
    'Retorno anual':     ret_m.mean() * n,

    # Volatilidad anualizada: σ_mensual × √12
    # Propiedad estadística: si los retornos son iid, la varianza escala linealmente
    # Por lo tanto: σ_anual = σ_mensual × √12
    'Volatilidad anual': ret_m.std() * np.sqrt(n),

    # Sharpe Ratio: (retorno promedio - tasa libre de riesgo) / volatilidad
    # Mide cuánto retorno en exceso se obtiene por unidad de riesgo asumido
    # Un Sharpe > 1 se considera bueno; > 2 es muy bueno
    'Sharpe':            (ret_m.mean() - Rf_m.mean()) * n / (ret_m.std() * np.sqrt(n)),

    # Sesgo (skewness): distribuciones con sesgo negativo tienen mayor riesgo de caída brusca
    'Skewness':          ret_m.skew(),

    # Curtosis en exceso: > 0 indica colas más gruesas que la normal (más eventos extremos)
    # Esto es relevante para el cálculo de VaR y Expected Shortfall
    'Kurtosis':          ret_m.kurt(),

    # Maximum Drawdown: peor caída desde un máximo histórico hasta el mínimo siguiente
    # raw.cummax(): precio máximo acumulado hasta cada fecha
    # raw / raw.cummax() - 1: caída porcentual desde el máximo (siempre ≤ 0)
    # .min(): el punto de mayor caída en toda la historia
    'Max Drawdown':      (raw / raw.cummax() - 1).min()
})

# Ordenar de mayor a menor Sharpe para identificar los mejores activos
stats_activos.sort_values('Sharpe', ascending=False)

In [ ]:
# ── Mapa de calor de correlaciones ────────────────────────────────────────
# La correlación mide el grado de movimiento conjunto entre dos activos
# Valores cercanos a 1: se mueven juntos (baja diversificación)
# Valores cercanos a 0: independientes (buena diversificación)
# Valores negativos: se mueven en sentido opuesto (cobertura natural)

# .corr(): calcula la matriz de correlaciones de Pearson entre todas las columnas
corr = ret_m.corr()

fig, ax = plt.subplots(figsize=(14, 10))

# Crear una máscara triangular para mostrar solo la mitad inferior del mapa
# (la parte superior es simétrica, por lo que es redundante)
# np.triu: devuelve True en la parte superior triangular
# np.ones_like: crea una matriz de unos con la misma forma que 'corr'
# dtype=bool: convierte a valores booleanos (True/False)
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr,
    mask=mask,         # Ocultar triángulo superior
    annot=True,        # Mostrar el valor numérico en cada celda
    fmt='.2f',         # Formato: 2 decimales (ej. 0.73)
    cmap='RdYlGn',     # Escala de colores: rojo (correlación baja/negativa) → verde (alta)
    center=0,          # El color neutro (amarillo) está en correlación = 0
    vmin=-1, vmax=1,   # Rango completo de correlaciones posibles
    linewidths=0.5,    # Grosor de las líneas divisorias entre celdas
    ax=ax
)
ax.set_title('Matriz de Correlación de Retornos Mensuales', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Distribución de retornos vs. distribución normal ─────────────────────
# Uno de los supuestos clásicos de las finanzas (ej. Black-Scholes) es que
# los retornos siguen una distribución normal. En la práctica, esto rara vez se cumple.
# Este análisis visual y estadístico muestra el grado de desviación de la normalidad.

ncols = 4    # Número de columnas en la grilla de subgráficos
# Calcular el número de filas necesario para mostrar todos los activos
# np.ceil: redondear hacia arriba (si no son divisibles exactamente)
nrows = int(np.ceil(len(ret_m.columns) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
axes = axes.flatten()    # Convertir la grilla 2D de ejes en una lista 1D para iterar fácilmente

for i, ticker in enumerate(ret_m.columns):
    r = ret_m[ticker].dropna()    # Retornos del activo sin valores faltantes

    # Histograma de retornos observados
    # bins=30: número de barras del histograma
    # density=True: normaliza el histograma para que el área total sea 1 (como una PDF)
    # alpha=0.6: transparencia del 40% para que se vea la curva normal detrás
    axes[i].hist(r, bins=30, density=True, alpha=0.6, color='steelblue')

    # Curva normal teórica ajustada a los mismos parámetros (media y desviación estándar)
    xmin, xmax = axes[i].get_xlim()           # Rango del eje X del histograma
    x = np.linspace(xmin, xmax, 100)          # 100 puntos equiespaciados en ese rango
    # sp_stats.norm.pdf: función de densidad de probabilidad de la distribución normal
    axes[i].plot(x, sp_stats.norm.pdf(x, r.mean(), r.std()), 'r--', linewidth=1.5)

    # Test de Jarque-Bera: prueba estadística de normalidad
    # H0: los datos siguen una distribución normal
    # Si p < 0.05: se rechaza la normalidad (los retornos NO son normales)
    jb_stat, jb_p = sp_stats.jarque_bera(r)

    # Mostrar ticker y p-valor del test en el título de cada subgráfico
    axes[i].set_title(f'{ticker}  (JB p={jb_p:.3f})', fontsize=9)
    axes[i].tick_params(labelsize=7)

# Ocultar los subgráficos sobrantes si el número de activos no es múltiplo de ncols
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribución de Retornos Mensuales vs. Normal Teórica (línea roja)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Exportación de Datos Procesados

Los datos limpios y alineados se guardan en `data/processed/` para ser reutilizados en los notebooks siguientes sin necesidad de repetir la descarga y el procesamiento.

In [ ]:
import os    # Módulo estándar de Python para interactuar con el sistema operativo (crear carpetas, rutas)

# Crear la carpeta si no existe; exist_ok=True evita error si ya está creada
os.makedirs('../data/processed', exist_ok=True)

# Guardar retornos mensuales alineados con FF6
# .to_csv(): exporta el DataFrame como archivo CSV (valores separados por coma)
ret_m.to_csv('../data/processed/retornos_mensuales.csv')

# Guardar los 6 factores FF6 alineados con los retornos
ff6_m.to_csv('../data/processed/factores_ff6_mensuales.csv')

# Guardar retornos diarios (se usarán en los modelos GARCH del notebook 04)
ret_daily.to_csv('../data/processed/retornos_diarios.csv')

print('Archivos exportados exitosamente a data/processed/:')
print(f'  retornos_mensuales.csv        → {ret_m.shape[0]} meses × {ret_m.shape[1]} activos')
print(f'  factores_ff6_mensuales.csv    → {ff6_m.shape[0]} meses × {ff6_m.shape[1]} factores')
print(f'  retornos_diarios.csv          → {ret_daily.shape[0]} días  × {ret_daily.shape[1]} activos')

---

## Resumen del notebook

| Paso | Descripción | Output |
|---|---|---|
| 1 | Descarga factores FF6 (French Library) | `ff6_m`: DataFrame 7 columnas |
| 2 | Descarga precios ajustados (yfinance) | `raw`: DataFrame de precios diarios |
| 3 | Cálculo de retornos diarios y mensuales | `ret_daily`, `ret_monthly` |
| 4 | Estadísticas descriptivas y visualizaciones | Tablas y gráficos exploratorios |
| 5 | Exportación a CSV | 3 archivos en `data/processed/` |

## Próximos pasos

- **Notebook 02 — Regresiones FF6:** Estimar los betas (sensibilidades) de cada activo frente a los 6 factores mediante regresiones de series de tiempo (OLS).
- **Notebook 03 — Matriz de Covarianza:** Construir la matriz de covarianza factorial vs. muestral.

---
*Universidad Diego Portales — Magíster en Finanzas*